# Predicción probabilística de fraude con tarjetas de crédito

**Informe final — LDS1071 · Kaggle Playground Series S6E11**

**Pregunta:** ¿qué combinación de representación de datos, algoritmo e hiperparámetros maximiza ROC AUC para predecir fraude?

El estudio sigue una secuencia única: análisis exploratorio → preprocesamiento → ingeniería de variables → búsqueda anidada → ensambles → evaluación. Cada resultado nuevo procede de las mismas particiones externas.

![Flujo metodológico](experiments/experiment023/flujo_metodologico.svg)


## 0. Configuración y reproducibilidad

El protocolo experiment023 se ejecuta por defecto; los checkpoints privados compatibles permiten reanudar automáticamente. Los experimentos previos están archivados y no se mezclan con esta comparación. Reinicie el kernel y ejecute todas las celdas. Los resultados se generan localmente; no se publican datos ni predicciones individuales.


In [ ]:
from __future__ import annotations
import os, zipfile, json, io, warnings, time, sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display, Markdown
from sklearn.model_selection import StratifiedKFold, train_test_split, GridSearchCV
from sklearn.preprocessing import OneHotEncoder, StandardScaler, SplineTransformer, RobustScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans, DBSCAN
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, IsolationForest
from sklearn.metrics import roc_auc_score, average_precision_score, roc_curve, precision_recall_curve, log_loss, brier_score_loss

SEMILLA = 42
REANUDAR = True  # Recuperar únicamente checkpoints compatibles de experiment023.
N_PARTICIONES = 5
N_BAGGING = 5
MUESTRA_GRAFICAS = 20_000
RAIZ = Path.cwd()
if not (RAIZ / 'credit_card_fraud.ipynb').exists() and (RAIZ / 'final_project' / 'credit_card_fraud.ipynb').exists():
    RAIZ = RAIZ / 'final_project'
if not (RAIZ / 'credit_card_fraud.ipynb').exists():
    # Permitir ejecutar la versión descargada aislada junto a sus archivos de soporte.
    RAIZ = Path.cwd()
DATOS_AGREGADOS = RAIZ / 'experiments' / 'experiment016' / 'resultados'
if not DATOS_AGREGADOS.is_dir() and (RAIZ / 'resultados').is_dir():
    DATOS_AGREGADOS = RAIZ / 'resultados'
SALIDA = RAIZ / 'outputs' / 'informe_experiment023_es'
FIGURAS = SALIDA / 'figuras'
FIGURAS.mkdir(parents=True, exist_ok=True)
sns.set_theme(style='whitegrid', palette='dark', context='notebook')
plt.rcParams.update({'figure.dpi':110, 'savefig.dpi':150, 'axes.titlesize':12, 'axes.titleweight':'semibold'})
COLORES = sns.color_palette('dark')

def cerrar_figura(fig, nombre):
    fig.tight_layout()
    fig.savefig(FIGURAS / f'{nombre}.png', bbox_inches='tight', dpi=150)
    plt.show(); plt.close(fig)

DICCIONARIO = json.loads((DATOS_AGREGADOS/'diccionario_variables_es.json').read_text(encoding='utf-8')) if (DATOS_AGREGADOS/'diccionario_variables_es.json').exists() else {}
def etiqueta(col):
    return DICCIONARIO.get(col, str(col).replace('_',' ').capitalize())

print('Ubicación del proyecto:', RAIZ.resolve())
print('Protocolo factorial experiment023 | Reanudar:', REANUDAR)
print('Versiones: Python', sys.version.split()[0], '| pandas', pd.__version__)

def anotaciones_matriz(m):
    total=np.sum(m)
    return np.array([[f'{int(valor):,}\n({valor/total:.2%})' for valor in fila] for fila in m],dtype=object)

## 1. Análisis exploratorio de datos (EDA)

### 1.1 Variables, integridad y prevalencia de fraude

Se examinan tipos de variables, distribución del objetivo, faltantes y desbalance de clases antes de ajustar modelos. El identificador `id` se excluye de los predictores.


In [ ]:
def leer_csv(nombre):
    for ruta in [RAIZ/'data'/nombre, RAIZ/nombre, Path('/kaggle/input/playground-series-s6e11')/nombre]:
        if ruta.is_file():
            return pd.read_csv(ruta)
    for ruta in [RAIZ/'playground-series-s6e11.zip', RAIZ/'data'/'playground-series-s6e11.zip', RAIZ.parent/'playground-series-s6e11.zip']:
        if ruta.is_file():
            with zipfile.ZipFile(ruta) as archivo:
                with archivo.open(nombre) as fichero:
                    return pd.read_csv(fichero)
    raise FileNotFoundError(f'No se encontró {nombre}. Coloque el ZIP original de Kaggle en la raíz.')

entrenamiento = leer_csv('train.csv')
prueba = leer_csv('test.csv')
plantilla_envio = leer_csv('sample_submission.csv')
OBJETIVO, IDENTIFICADOR = 'is_fraud', 'id'
assert OBJETIVO not in prueba and plantilla_envio[IDENTIFICADOR].equals(prueba[IDENTIFICADOR])
assert entrenamiento[IDENTIFICADOR].is_unique and prueba[IDENTIFICADOR].is_unique
assert entrenamiento[OBJETIVO].notna().all() and set(entrenamiento[OBJETIVO].unique()) == {0.,1.}
variables = [c for c in entrenamiento.columns if c not in [OBJETIVO, IDENTIFICADOR]]
assert set(variables) == set(prueba.columns) - {IDENTIFICADOR}
numericas = entrenamiento[variables].select_dtypes(include='number').columns.tolist()
categoricas = entrenamiento[variables].select_dtypes(include=['object','category']).columns.tolist()
logicas = entrenamiento[variables].select_dtypes(include='bool').columns.tolist()
y = entrenamiento[OBJETIVO].astype(int).to_numpy()
assert len(variables)==24 and len(numericas)==13 and len(categoricas)==5 and len(logicas)==6
print(f'Entrenamiento: {len(entrenamiento):,} | Prueba: {len(prueba):,} | Variables: {len(variables)}')
print('Fraude:',int(y.sum()),f'({y.mean():.2%})')
display(entrenamiento.head(3))
resumen_variables = pd.DataFrame({'variable':variables,'nombre_en_español':[etiqueta(c) for c in variables],
 'tipo':[str(entrenamiento[c].dtype) for c in variables], 'faltantes':[int(entrenamiento[c].isna().sum()) for c in variables]})
display(resumen_variables)

### 1.2 Distribuciones y desbalance

La exploración utiliza totales completos para prevalencia, tasas por categoría e interacciones; en gráficos con muchos puntos se toma una muestra aleatoria de tamaño declarado, para evitar saturación. **No se elimina automáticamente ninguna observación extrema**: una transacción inusual puede contener información relevante para detectar fraude.

### 2.1 Desbalance, valores faltantes y distribución numérica


In [ ]:
muestra = entrenamiento.sample(n=min(MUESTRA_GRAFICAS,len(entrenamiento)), random_state=SEMILLA).copy()
fig, ejes = plt.subplots(1,2,figsize=(13,4.7))
conteos = entrenamiento[OBJETIVO].value_counts().reindex([0.,1.],fill_value=0)
sns.barplot(x=['Legítimas','Fraudulentas'],y=conteos.values,hue=['Legítimas','Fraudulentas'],
            palette=[COLORES[0],COLORES[3]],legend=False,ax=ejes[0])
for k,v in enumerate(conteos): ejes[0].text(k,v*1.01,f'{v:,} ({v/len(entrenamiento):.2%})',ha='center',fontsize=10)
ejes[0].set(title='Distribución observada de clases',xlabel='Clase',ylabel='Número de transacciones',ylim=(0,conteos.max()*1.14))
faltantes=entrenamiento[variables].isna().mean()*100
sns.barplot(x=faltantes.values,y=[etiqueta(c) for c in faltantes.index],color=COLORES[0],ax=ejes[1])
ejes[1].set(title='Datos faltantes por variable',xlabel='Porcentaje de valores faltantes',ylabel='Variable',xlim=(0,max(1,float(faltantes.max())*1.12)))
cerrar_figura(fig,'01_desbalance_y_faltantes')

cols=[c for c in ['amount_usd','hours_since_last_txn','distance_from_home_km','velocity_score','merchant_risk_score','account_balance_usd'] if c in muestra]
fig,ejes=plt.subplots(2,3,figsize=(16,8.8))
for ax,col in zip(ejes.ravel(),cols):
    sns.histplot(data=muestra,x=col,bins=42,color=COLORES[0],ax=ax)
    if col in ('amount_usd','account_balance_usd') and (muestra[col] > 0).all(): ax.set_xscale('log')
    ax.set(title=etiqueta(col),xlabel=etiqueta(col),ylabel='Transacciones (muestra)')
for ax in ejes.ravel()[len(cols):]: ax.axis('off')
fig.suptitle('Distribuciones de variables numéricas (muestra aleatoria)',y=1.01,fontweight='bold')
cerrar_figura(fig,'02_distribuciones_numericas')

### 1.3 Correlaciones y variables categóricas

La correlación de Pearson mide asociación lineal; Spearman evalúa ordenamientos monótonos. La ausencia de correlación alta **no** descarta interacciones ni dependencias no lineales. Las tasas por categoría se calculan sobre el **conjunto de entrenamiento completo**, no sobre la muestra empleada para visualizar.


In [ ]:
matriz=entrenamiento[numericas+logicas].astype(float)
fig,ax=plt.subplots(figsize=(12,10))
sns.heatmap(matriz.corr(method='spearman'),cmap='vlag',center=0,vmin=-1,vmax=1,
            xticklabels=[etiqueta(c) for c in matriz.columns],
            yticklabels=[etiqueta(c) for c in matriz.columns],ax=ax,
            cbar_kws={'label':'Correlación de Spearman'})
ax.set_title('Asociación monótona entre variables numéricas y lógicas')
cerrar_figura(fig,'03_correlacion_spearman')

fig,ejes=plt.subplots(1,2,figsize=(15.3,5.8))
for ax,col in zip(ejes,['merchant_category','channel']):
    g=entrenamiento.groupby(col,dropna=False)[OBJETIVO].agg(['mean','size']).sort_values('mean',ascending=False)
    g['nombre']=[f'{cat} (n={n:,})' for cat,n in zip(g.index,g['size'])]
    sns.barplot(data=g,x='mean',y='nombre',color=COLORES[0],ax=ax)
    ax.axvline(np.mean(y),linestyle='--',color=COLORES[3],label='Prevalencia total')
    ax.set(xlabel='Proporción observada de fraude',ylabel='',title=f'Riesgo por {etiqueta(col).lower()}')
    ax.legend(title='Referencia')
fig.suptitle('Tasas de fraude según categoría y canal (entrenamiento completo)',y=1.01,fontweight='bold')
cerrar_figura(fig,'04_fraude_por_categorias')

### 1.4 No linealidades e interacciones

Una transformación que retenga el 90 % de la varianza no necesariamente preservará los contrastes asociados al fraude. Antes de modelar evaluamos tasas de fraude por cuantiles para identificar posibles umbrales y relaciones no monótonas. Las tasas marginales no son efectos causales ni sustituyen al ajuste multivariante.


In [ ]:
analizar=['amount_usd','hours_since_last_txn','velocity_score','merchant_risk_score']
fig,ejes=plt.subplots(2,2,figsize=(14.4,9))
for ax,col in zip(ejes.ravel(),analizar):
    intervalos=pd.qcut(entrenamiento[col],q=12,duplicates='drop')
    tabla=entrenamiento.groupby(intervalos,observed=True)[OBJETIVO].agg(['mean','size']).reset_index()
    sns.lineplot(x=np.arange(1,len(tabla)+1),y=100*tabla['mean'],marker='o',color=COLORES[0],ax=ax)
    ax.axhline(100*np.mean(y),linestyle='--',color=COLORES[3],label='Prevalencia total')
    ax.set(title=etiqueta(col),xlabel='Intervalo por cuantiles (de menor a mayor)',ylabel='Operaciones fraudulentas (%)')
    ax.legend()
fig.suptitle('Relaciones marginales no lineales con el fraude',fontweight='bold',y=1.01)
cerrar_figura(fig,'05_tasas_por_cuantiles')

pares=[('merchant_risk_score','velocity_score'),('amount_usd','distance_from_home_km')]
fig,ejes=plt.subplots(1,2,figsize=(15,6.1))
for ax,(a,b) in zip(ejes,pares):
    ga=pd.qcut(entrenamiento[a],q=5,labels=False,duplicates='drop')
    gb=pd.qcut(entrenamiento[b],q=5,labels=False,duplicates='drop')
    t=entrenamiento.groupby([ga.rename('a'),gb.rename('b')])[OBJETIVO].mean().unstack()*100
    sns.heatmap(t,annot=True,fmt='.2f',cmap='crest',ax=ax,cbar_kws={'label':'Fraude observado (%)'})
    ax.set(xlabel=f'{etiqueta(b)} (quintil)',ylabel=f'{etiqueta(a)} (quintil)',title='Tasa conjunta por quintiles')
fig.suptitle('Interacciones bivariantes en el conjunto completo',fontweight='bold',y=1.01)
cerrar_figura(fig,'06_interacciones_quintiles')

### 1.5 Patrones temporales y anomalías

Una hora del día es una variable periódica: 23:00 y 00:00 están próximas pese a tener valores numéricos distantes. Se consideran transformaciones trigonométricas como posibilidad de ingeniería de variables. Para los extremos se emplean cercas de 3×IQR únicamente como **diagnóstico descriptivo**, junto con un detector no supervisado (Isolation Forest) sobre una muestra.


In [ ]:
fig,ejes=plt.subplots(1,2,figsize=(14,4.8))
for ax,col in zip(ejes,['time_of_day_hour','day_of_week']):
    tasa=entrenamiento.groupby(col)[OBJETIVO].agg(['mean','size'])
    sns.barplot(x=tasa.index,y=100*tasa['mean'],color=COLORES[0],ax=ax)
    ax.axhline(100*np.mean(y),linestyle='--',color=COLORES[3],label='Prevalencia total')
    ax.set(xlabel=etiqueta(col),ylabel='Porcentaje de fraude',title='Variación temporal del riesgo')
    ax.legend()
cerrar_figura(fig,'07_variacion_temporal')

extremos=[]
for c in numericas:
    q1,q3=entrenamiento[c].quantile([.25,.75]); iq=q3-q1
    if iq==0:continue
    flag=(entrenamiento[c]<q1-3*iq)|(entrenamiento[c]>q3+3*iq)
    extremos.append({'Variable':etiqueta(c),'Proporción extrema':float(flag.mean()),'Tasa de fraude entre extremos':float(entrenamiento.loc[flag,OBJETIVO].mean()) if flag.any() else np.nan})
tabla_extremos=pd.DataFrame(extremos).sort_values('Proporción extrema',ascending=False)
fig,ax=plt.subplots(figsize=(12,6))
sns.barplot(data=tabla_extremos.head(9),x='Proporción extrema',y='Variable',color=COLORES[3],ax=ax)
ax.set(title='Valores fuera de las cercas conservadoras de 3×IQR',xlabel='Proporción de operaciones',ylabel='Variable')
cerrar_figura(fig,'08_extremos_univariados')

if len(muestra)>500:
    base_anom=muestra[numericas+logicas].astype(float).replace([np.inf,-np.inf],np.nan)
    base_anom=base_anom.fillna(base_anom.median())
    scaler_anom=RobustScaler().fit_transform(base_anom)
    iso=IsolationForest(n_estimators=90,max_samples=256,contamination=0.02,random_state=SEMILLA,n_jobs=-1)
    score=-iso.fit(scaler_anom).decision_function(scaler_anom)
    m=muestra[[OBJETIVO]].copy();m['índice de anomalía']=score
    m['clase']=m[OBJETIVO].map({0.:'Legítima',1.:'Fraudulenta'})
    fig,ax=plt.subplots(figsize=(11,4.7))
    sns.histplot(data=m,x='índice de anomalía',hue='clase',stat='density',common_norm=False,
                 bins=40,element='step',fill=False,ax=ax,palette={'Legítima':COLORES[0],'Fraudulenta':COLORES[3]})
    ax.set(title='Anomalía multivariante: distribución según clase (muestra)',xlabel='Puntuación de anomalía (mayor = más atípica)',ylabel='Densidad por clase')
    cerrar_figura(fig,'09_anomalias_isolation_forest')

### 1.6 Comparación de entrenamiento y prueba

Como las etiquetas de la prueba son desconocidas, solo se comparan las distribuciones de los predictores. Una diferencia de distribuciones entre `train` y `test` podría comprometer la generalización; **no** implica que exista una etiqueta de fraude disponible en la prueba.


In [ ]:
variables_traspaso=['amount_usd','velocity_score','merchant_risk_score']
fig,ejes=plt.subplots(1,3,figsize=(16,4.7))
for ax,c in zip(ejes,variables_traspaso):
    a=entrenamiento[c].sample(n=min(12000,len(entrenamiento)),random_state=SEMILLA).reset_index(drop=True)
    b=prueba[c].sample(n=min(12000,len(prueba)),random_state=SEMILLA).reset_index(drop=True)
    sns.ecdfplot(a,ax=ax,color=COLORES[0],label='Entrenamiento')
    sns.ecdfplot(b,ax=ax,color=COLORES[1],label='Prueba')
    ax.set(title=etiqueta(c),xlabel=etiqueta(c),ylabel='Proporción acumulada')
    ax.legend()
fig.suptitle('Comparación no supervisada de las distribuciones',fontweight='bold',y=1.01)
cerrar_figura(fig,'10_desplazamiento_entrenamiento_prueba')

## 2. Preprocesamiento e ingeniería de variables

### 2.1 Imputación, one-hot encoding y estandarización

Se conserva una versión sin escalado de los predictores numéricos y otra con $z_j=(x_j-\mu_j)/s_j$. Las variables booleanas se codifican 0/1 y las categóricas mediante one-hot encoding en ambas ramas. Las medias, desviaciones, imputaciones y categorías se estiman solo en el entrenamiento de cada split. Las categorías desconocidas se ignoran.

La geometría para K-means y DBSCAN usa siempre la rama estandarizada. Esta demostración sobre una muestra solo ilustra el procedimiento; no se reutiliza en la validación cruzada.


In [ ]:
pre_visual = ColumnTransformer([
    ('num', SimpleImputer(strategy='median'), numericas),
    ('bool', 'passthrough', logicas),
    ('cat', Pipeline([('imputar',SimpleImputer(strategy='most_frequent')),
                      ('onehot',OneHotEncoder(handle_unknown='ignore',sparse_output=False))]),categoricas)
], sparse_threshold=0)
X_visual=muestra[variables].assign(**{c:muestra[c].astype(float) for c in logicas})
M_raw=pre_visual.fit_transform(X_visual).astype(float)
M=M_raw.copy()
M[:,:len(numericas)]=StandardScaler().fit_transform(M_raw[:,:len(numericas)])
display(pd.DataFrame({'Variable':numericas,'Media original':M_raw[:,:len(numericas)].mean(0),
                     'Desviación original':M_raw[:,:len(numericas)].std(0),
                     'Media estandarizada':M[:,:len(numericas)].mean(0),
                     'Desviación estandarizada':M[:,:len(numericas)].std(0)}).round(5))
print('Dimensiones después de one-hot:', M.shape)


### 2.2 PCA y t-SNE: geometría de los predictores

PCA retiene el 90 % de la varianza para construir las distancias de agrupamiento. Las proyecciones siguientes son ilustrativas; t-SNE no se usa como predictor. La muestra enriquecida de fraude en t-SNE no representa la prevalencia poblacional.


In [ ]:
pca_v=PCA(n_components=.90,svd_solver='full',random_state=SEMILLA).fit(M)
pca_total=PCA(n_components=min(30,M.shape[1]),svd_solver='full',random_state=SEMILLA).fit(M)
fig,ejes=plt.subplots(1,2,figsize=(14.5,4.8))
ejes[0].plot(np.arange(1,len(pca_total.explained_variance_ratio_)+1),pca_total.explained_variance_ratio_.cumsum(),marker='o',markersize=3,color=COLORES[0])
ejes[0].axhline(.90,ls='--',color=COLORES[3],label='90 % de varianza')
ejes[0].set(xlabel='Número de componentes',ylabel='Varianza acumulada',title='PCA ilustrativa: varianza explicada')
ejes[0].legend()
XY=PCA(n_components=2,random_state=SEMILLA).fit_transform(M)
seleccion=np.random.default_rng(SEMILLA).choice(len(XY),size=min(5000,len(XY)),replace=False)
clases=muestra.iloc[seleccion][OBJETIVO].values.astype(int)
for val,color,nombre in [(0,COLORES[0],'Legítima'),(1,COLORES[3],'Fraudulenta')]:
    mask=clases==val
    ejes[1].scatter(XY[seleccion[mask],0],XY[seleccion[mask],1],s=8,alpha=.48,color=color,label=nombre,rasterized=True)
ejes[1].set(xlabel='Primera componente',ylabel='Segunda componente',title='Proyección PCA en dos dimensiones')
ejes[1].legend(title='Clase observada')
fig.suptitle('Reducción dimensional para explorar la estructura de los predictores',fontweight='bold',y=1.01)
cerrar_figura(fig,'11_pca_90_varianza')
print('PCA ilustrativa: componentes para 90 % de varianza:',pca_v.n_components_,'de',M.shape[1])

In [ ]:
from sklearn.manifold import TSNE
generador_tsne=np.random.default_rng(SEMILLA)
pos=np.flatnonzero(muestra[OBJETIVO].to_numpy()==1)
neg=np.flatnonzero(muestra[OBJETIVO].to_numpy()==0)
idx=np.r_[generador_tsne.choice(neg,size=min(1500,len(neg)),replace=False),
          generador_tsne.choice(pos,size=min(270,len(pos)),replace=False)]
coords=TSNE(n_components=2,perplexity=35,init='pca',learning_rate='auto',max_iter=800,random_state=SEMILLA).fit_transform(M[idx])
clases=muestra.iloc[idx][OBJETIVO].to_numpy()
fig,ax=plt.subplots(figsize=(9,5.6))
for clase,color,nombre in [(0,COLORES[0],'Legítima'),(1,COLORES[3],'Fraudulenta')]:
    mask=clases==clase
    ax.scatter(coords[mask,0],coords[mask,1],color=color,s=15,alpha=.5,label=f'{nombre} (n={mask.sum():,})',rasterized=True)
ax.set(title='t-SNE: vecindades locales (muestra enriquecida)',xlabel='Coordenada t-SNE 1',ylabel='Coordenada t-SNE 2')
ax.legend(title='Clase observada')
cerrar_figura(fig,'11b_tsne')

### 2.3 K-means y DBSCAN

K-means con cinco centroides aporta cinco distancias y la mínima. DBSCAN (min_samples=5, radio igual al cuantil 90 % de la distancia al quinto vecino de una referencia) aporta la distancia al núcleo más cercano y un indicador de ruido. Los ocho predictores derivados se añaden sin eliminar los originales. PCA y ambos agrupamientos se ajustan de nuevo en cada split. Se usan referencias de hasta 15 000 y 2 300 observaciones de entrenamiento, respectivamente. No se vuelve a seleccionar el número de clusters con los resultados externos.

Las gráficas siguientes son diagnósticos descriptivos nuevos de la muestra, no criterios de selección supervisada.


In [ ]:
from sklearn.metrics import silhouette_score
from sklearn.neighbors import NearestNeighbors
rng=np.random.default_rng(SEMILLA)
ref=M[rng.choice(len(M),size=min(2300,len(M)),replace=False)]
z=PCA(n_components=.90,svd_solver='full').fit_transform(ref)
filas=[]
for k in range(2,11):
    km=KMeans(n_clusters=k,n_init=10,random_state=SEMILLA).fit(z)
    filas.append({'k':k,'inercia':km.inertia_,'silueta':silhouette_score(z,km.labels_)})
d_k=pd.DataFrame(filas)
d5=NearestNeighbors(n_neighbors=5).fit(z).kneighbors(z)[0][:,-1]
filas=[]
for q in (.50,.70,.80,.90,.95,.99):
    eps=max(float(np.quantile(d5,q)),1e-6)
    labels=DBSCAN(eps=eps,min_samples=5).fit_predict(z)
    filas.append({'cuantil':q,'epsilon':eps,'ruido':float((labels==-1).mean()),
                  'clusters':len(set(labels)-{-1})})
d_db=pd.DataFrame(filas)
display(d_k.round(4));display(d_db.round(4))
fig,ejes=plt.subplots(1,3,figsize=(15,4.5))
ejes[0].plot(d_k.k,d_k.inercia,marker='o');ejes[0].axvline(5,ls='--',color=COLORES[3])
ejes[0].set(title='K-means: inercia',xlabel='k',ylabel='Inercia')
ejes[1].plot(d_k.k,d_k.silueta,marker='o');ejes[1].axvline(5,ls='--',color=COLORES[3])
ejes[1].set(title='K-means: silueta',xlabel='k',ylabel='Silueta')
ejes[2].plot(d_db.cuantil,d_db.ruido,marker='o');ejes[2].axvline(.90,ls='--',color=COLORES[3])
ejes[2].set(title='DBSCAN: ruido',xlabel='Cuantil para epsilon',ylabel='Fracción de ruido')
cerrar_figura(fig,'12_diagnostico_clusters')


### 2.4 Espacio factorial de representaciones

| Representación | Numéricas estandarizadas | One-hot | K-means y DBSCAN |
|---|---|---|---|
| raw | No | Sí | No |
| standardized | Sí | Sí | No |
| raw_kmeans_dbscan | No | Sí | Sí |
| standardized_kmeans_dbscan | Sí | Sí | Sí |

Las distancias añadidas se mantienen sin escalado en la rama raw y se estandarizan en la rama standardized; el indicador de ruido se conserva binario. PCA y splines no constituyen candidatos adicionales en esta comparación de cuatro representaciones.


## 3. Clasificadores individuales: búsqueda anidada común

Cada familia usa el producto cartesiano de las cuatro representaciones y su grilla propia. Se conservan los mejores hiperparámetros de cada representación y el selector adaptativo que elige representación e hiperparámetros por AUC interna en cada fold. Las cinco particiones externas y los tres splits internos (muestra estratificada de hasta 36 000 filas por fold externo) son comunes. Ninguna etiqueta de validación externa participa en la selección.

### 3.1 Modelos y supuestos

Regresión logística (L2 y ponderación), bosque aleatorio, XGBoost y CatBoost. Las relaciones no lineales pueden ser capturadas por árboles; la logística sin escalado puede presentar problemas de convergencia, por lo que se permiten hasta 2 000 iteraciones y se conservan sus advertencias.


In [ ]:
supuestos=pd.DataFrame([
    ['Respuesta binaria','Verificado','La etiqueta toma los valores 0 y 1.'],
    ['Predictores observados / faltantes','Verificado','Se auditan los faltantes y la imputación se ajusta por pliegue.'],
    ['Linealidad en el logit','No garantizada','Se detectaron patrones no lineales; se contrastan logística y árboles.'],
    ['Independencia por cliente','No verificable','No se proporciona un identificador de cliente longitudinal.'],
    ['Colinealidad / redundancia','Mitigada','Codificación one-hot completa y regularización L2.'],
    ['Separación completa','No demostrada ausente','Una revisión univariada no descarta separación multivariada.'],
    ['Tamaño de muestra','Adecuado para empezar','Hay 11 689 fraudes; importa también la rareza de combinaciones de categorías.'],
    ['Calibración de probabilidades','Se evalúa con OOF','Pérdida logarítmica, Brier y curvas de calibración.'],
],columns=['Condición','Situación','Evidencia y limitación'])
display(supuestos)

### 3.2 Grillas y ejecución

XGBoost utiliza 36 configuraciones (6 profundidades × 2 tasas de aprendizaje × 3 valores de min_child_weight) en el modelo individual, el meta-modelo y el corrector de logits. Se registra cada evaluación interna y se guardan los mejores parámetros, AUC externa y predicciones privadas por fold.


In [ ]:
from sklearn.model_selection import ParameterGrid
sys.path.insert(0,str(RAIZ/'experiments'/'experiment023'))
import protocolo as exp
exp.configurar(numericas,logicas,categoricas,N_BAGGING)
directorio=RAIZ/'outputs'/'experiment023'
X=entrenamiento[variables]; X_test=prueba[variables]
firma=exp.firma_datos(X,y,X_test,entrenamiento[IDENTIFICADOR],prueba[IDENTIFICADOR])
tabla_grillas=pd.DataFrame([
    {'Familia':f,'Representaciones':len(exp.REPRESENTACIONES),'Grilla':str(exp.GRILLAS_COMUNES[f]),
     'Configuraciones por representación':len(ParameterGrid(exp.GRILLAS_COMUNES[f])),
     'Configuraciones por fold':4*len(ParameterGrid(exp.GRILLAS_COMUNES[f]))}
    for f in exp.FAMILIAS_INDIVIDUALES])
display(tabla_grillas)
assert len(ParameterGrid(exp.GRILLA_BOOSTER))==36


In [ ]:
resultados_individuales,grilla_individual,seleccion_individual,estados=exp.protocolo_individual(
    X,y,X_test,directorio,firma,REANUDAR)
directorio.mkdir(parents=True,exist_ok=True)
grilla_individual.to_csv(directorio/'individual_grid.csv',index=False)
seleccion_individual.to_csv(directorio/'individual_selected.csv',index=False)
resumen_individual=exp.resumir_predicciones(resultados_individuales,y)
resumen_individual.to_csv(directorio/'individual_summary.csv',index=False)
display(resumen_individual.round(6))
display(seleccion_individual[seleccion_individual.familia.str.endswith('| adaptive')].round(6))
for i,f in enumerate(exp.FAMILIAS_INDIVIDUALES):
    fig,ax=plt.subplots(figsize=(11,4.5))
    sns.boxplot(data=grilla_individual[grilla_individual.familia==f],x='representacion',y='auc_interna',ax=ax,color=COLORES[0])
    ax.set(title=f+' · grilla interna',xlabel='Representación',ylabel='ROC AUC interna')
    cerrar_figura(fig,f'13_grilla_{i}')


## 4. Ensambles: mismas representaciones y particiones

En cada escenario fijo se combinan las cuatro familias ajustadas con esa representación. También se evalúa el escenario adaptativo, que usa la representación elegida por AUC interna para cada familia. Se comparan cuatro estrategias en los cinco escenarios:

- **Promedio:** media uniforme de las cuatro probabilidades.
- **Bagging:** cinco réplicas bootstrap de XGBoost con sus parámetros seleccionados; el preprocesador se ajusta en el entrenamiento externo.
- **Stacking:** meta-modelo XGBoost ajustado sobre cuatro probabilidades generadas por cross-fitting; 36 configuraciones. La selección de bases precede a la grilla del meta-modelo y condiciona su AUC interna, mientras el fold externo permanece retenido.
- **Corrección de logits:** mejor regresión logística del escenario, seguida de XGBoost con su logit como base_margin tanto en ajuste como en predicción. El booster utiliza la misma representación que la logística seleccionada. Se evalúan 36 configuraciones.

$$\hat p(x)=\sigma\big(f_{LR}(x)+g_{XGB}(x)\big).$$

No se presupone que el corrector supere a la logística ni que el agrupamiento mejore el resultado. Los veinte candidatos de ensamble y los veinte individuales se evalúan con predicciones OOF comparables.


In [ ]:
resultados_ensambles,grilla_ensambles,seleccion_ensambles=exp.protocolo_ensambles(
    X,y,X_test,estados,directorio,firma,REANUDAR)
resultados_comunes={**resultados_individuales,**resultados_ensambles}
assert len(resultados_comunes)==40
grilla_comun=pd.concat([grilla_individual,grilla_ensambles],ignore_index=True)
seleccion_comun=pd.concat([seleccion_individual,seleccion_ensambles],ignore_index=True)
resumen_comun=exp.guardar_agregados(directorio,grilla_comun,seleccion_comun,resultados_comunes,y)
display(exp.resumir_predicciones(resultados_ensambles,y).round(6))
display(seleccion_ensambles.round(6))
for i,f in enumerate(('Stacking XGBoost','Logística + XGBoost')):
    fig,ax=plt.subplots(figsize=(11,4.5))
    sns.boxplot(data=grilla_ensambles[grilla_ensambles.familia==f],x='representacion',y='auc_interna',ax=ax,color=COLORES[0])
    ax.set(title=f+' · grilla interna',xlabel='Escenario',ylabel='ROC AUC interna')
    cerrar_figura(fig,f'14_grilla_ensamble_{i}')


## 5. Selección final y evaluación

### 5.1 Diez mejores candidatos por ROC AUC

Se ordenan los 40 candidatos por ROC AUC OOF descendente y se omiten duplicados exactos de predicciones OOF y prueba. Un selector adaptativo puede coincidir con una representación fija; no se cuentan dos veces las mismas predicciones. Las métricas del top 10 son descriptivas tras selección, no una prueba independiente.


In [ ]:
def ordenar_top10(resumen,predicciones):
    import hashlib
    vistos=set(); filas=[]
    for fila in resumen.sort_values(['roc_auc','familia'],ascending=[False,True],kind='stable').itertuples(index=False):
        f=fila.familia
        if f in predicciones:
            d=predicciones[f]
            h=hashlib.sha256()
            for k in ('oof','test'):
                h.update(np.ascontiguousarray(d[k],dtype=np.float64).tobytes())
            key=h.digest()
            if key in vistos: continue
            vistos.add(key)
        filas.append(f)
    tabla=resumen.set_index('familia').loc[filas[:10]].reset_index()
    tabla.insert(0,'puesto',np.arange(1,len(tabla)+1))
    return tabla

def graficar_top10(tabla,nombre):
    display(tabla.round(6))
    orden=tabla.iloc[::-1]
    fig,ax=plt.subplots(figsize=(13,6.5))
    pos=np.arange(len(orden))
    ax.scatter(orden.roc_auc,pos,s=85,color=COLORES[0])
    ax.set_yticks(pos,orden.familia)
    for j,v in enumerate(orden.roc_auc):
        ax.annotate(f'{v:.6f}',(v,j),xytext=(7,0),textcoords='offset points',va='center')
    mn,mx=orden.roc_auc.min(),orden.roc_auc.max()
    margen=max((mx-mn)*.08,.0001)
    ax.set_xlim(mn-margen,mx+3*margen)
    ax.set(title='Top 10 modelos por ROC AUC OOF',xlabel='ROC AUC OOF (eje ampliado)',ylabel='')
    cerrar_figura(fig,nombre)

top10=pd.DataFrame()
if resultados_comunes:
    top10=ordenar_top10(resumen_comun,resultados_comunes)
    assert len(top10)==10, 'No hay diez candidatos distintos con predicciones completas.'
    graficar_top10(top10,'comparacion_final_top10')
    top10.to_csv(SALIDA/'top10.csv',index=False)
    print('Finalista:',top10.iloc[0].familia)



### 5.2 Finalista y envío Kaggle

La primera posición del ranking determina el finalista. Se exportan probabilidades, sin umbral de clasificación. No se envía automáticamente a Kaggle.


In [ ]:
resultados_oof={f:resultados_comunes[f] for f in top10.familia}
traduccion_estrategias={f:f for f in resultados_oof}
seleccionada=str(top10.iloc[0].familia)
prob=np.asarray(resultados_oof[seleccionada]['test'],dtype=float)
assert prob.shape==(len(prueba),) and np.isfinite(prob).all() and ((prob>=0)&(prob<=1)).all()
envio=plantilla_envio[[IDENTIFICADOR]].copy()
envio[OBJETIVO]=prob
assert envio[IDENTIFICADOR].equals(prueba[IDENTIFICADOR])
envio.to_csv(SALIDA/'envio_final_kaggle.csv',index=False)
print('Finalista:',seleccionada,'| Filas:',len(envio))


### 5.3 ROC, precisión–exhaustividad y calibración

Todas las curvas se calculan con las predicciones OOF de los mismos diez candidatos seleccionados por ROC AUC. Se conserva el orden y el código de cada candidato.


In [ ]:
if resultados_oof:
    resultados_oof=dict(sorted(resultados_oof.items(),key=lambda t:roc_auc_score(y,t[1]['oof']),reverse=True))
    orden_pr=dict(sorted(resultados_oof.items(),key=lambda t:average_precision_score(y,t[1]['oof']),reverse=True))
    fig,ejes=plt.subplots(1,2,figsize=(14.5,5.5))
    for i,(nombre,datos) in enumerate(resultados_oof.items()):
        prob=datos['oof']; color=COLORES[i%len(COLORES)]
        fpr,tpr,_=roc_curve(y,prob)
        sel=np.unique(np.linspace(0,len(fpr)-1,min(2500,len(fpr)),dtype=int))
        fpr,tpr=fpr[sel],tpr[sel]
        prec,rec,_=precision_recall_curve(y,prob,drop_intermediate=True)
        sel=np.unique(np.linspace(0,len(prec)-1,min(2500,len(prec)),dtype=int))
        prec,rec=prec[sel],rec[sel]
        ejes[0].plot(fpr,tpr,color=color,alpha=1 if i==0 else .5,label=f'{traduccion_estrategias[nombre]} · {roc_auc_score(y,prob):.4f}')

    for i,(nombre,datos) in enumerate(orden_pr.items()):
        prec,rec,_=precision_recall_curve(y,datos['oof'],drop_intermediate=True)
        sel=np.unique(np.linspace(0,len(prec)-1,min(2500,len(prec)),dtype=int))
        color=COLORES[list(resultados_oof).index(nombre)%len(COLORES)]
        ejes[1].plot(rec[sel],prec[sel],color=color,alpha=1 if i==0 else .5,
                     label=f'{traduccion_estrategias[nombre]} · {average_precision_score(y,datos["oof"]):.4f}')
    ejes[0].plot([0,1],[0,1],ls='--',color='gray',label='Azar')
    ejes[0].set(title='Curvas ROC en predicciones OOF',xlabel='Tasa de falsos positivos',ylabel='Tasa de verdaderos positivos')
    ejes[1].axhline(np.mean(y),ls='--',color='gray',label='Prevalencia de fraude')
    ejes[1].set(title='Curvas precisión–exhaustividad',xlabel='Exhaustividad (sensibilidad)',ylabel='Precisión')
    for ax in ejes:ax.legend(fontsize=8,loc='best')
    cerrar_figura(fig,'15_curvas_roc_y_precision_exhaustividad')
    fig,ax=plt.subplots(figsize=(9,6))
    for i,(nombre,datos) in enumerate(resultados_oof.items()):
        tabla=pd.DataFrame({'real':y,'riesgo':datos['oof']})
        tabla['intervalo']=pd.qcut(tabla['riesgo'],q=15,duplicates='drop')
        grupo=tabla.groupby('intervalo',observed=True).agg(p_pred=('riesgo','mean'),p_real=('real','mean'))
        ax.plot(grupo['p_pred'],grupo['p_real'],marker='o',markersize=3,alpha=1 if i==0 else .5,label=traduccion_estrategias[nombre],color=COLORES[i%len(COLORES)])
    ax.plot([0,1],[0,1],color='black',ls='--',label='Calibración ideal')
    ax.set(title='Calibración observada por intervalos de riesgo OOF',xlabel='Probabilidad media estimada',ylabel='Frecuencia observada de fraude',xlim=(0,1),ylim=(0,1))
    ax.legend(fontsize=8)
    cerrar_figura(fig,'16_calibracion_oof')
else:
    print('Curvas y calibración omitidas por ausencia de vectores OOF privados.')

### 5.5 Determinación del umbral de alerta

Las probabilidades pueden convertirse en decisiones mediante `predicción ≥ umbral`. Reducir falsos negativos suele incrementar falsos positivos. Se estudia el umbral convencional 0.50, el que maximiza F1 y el **mayor umbral compatible con 98 % de exhaustividad** sobre datos de selección, solo como objetivo ilustrativo. Se reportan VP, VN, FP, FN, sus porcentajes, F1, precisión, recall, exactitud y ROC AUC.

Los umbrales de evaluación cruzada se eligen en cuatro folds y se aplican al quinto; esto reduce optimismo pero no equivale a validación independiente ni resuelve dependencias por cliente.


In [ ]:
from sklearn.metrics import (confusion_matrix,f1_score,accuracy_score,precision_score,recall_score,
                             balanced_accuracy_score)
OBJETIVO_RECALL=0.98

def elegir_umbral(y_real,p,objetivo=OBJETIVO_RECALL):
    positivos=np.sort(np.asarray(p)[np.asarray(y_real)==1])
    if not len(positivos):raise ValueError('No hay observaciones fraudulentas')
    return float(positivos[max(0,int(np.floor((1-objetivo)*len(positivos))))])

def metricas_alerta(y_real,p,umbral,nombre,politica):
    b=np.asarray(p)>=umbral
    vn,fp,fn,vp=confusion_matrix(y_real,b,labels=[0,1]).ravel()
    return {'Estrategia':nombre,'Política':politica,'Umbral':umbral,
            'VP':int(vp),'FN':int(fn),'FP':int(fp),'VN':int(vn),
            'Recall':recall_score(y_real,b),'Precisión':precision_score(y_real,b,zero_division=0),
            'F1':f1_score(y_real,b,zero_division=0),'Exactitud':accuracy_score(y_real,b),
            'ROC AUC':roc_auc_score(y_real,p),'Tasa de falsas alarmas':fp/(fp+vn)}

predicciones={k:v['oof'] for k,v in resultados_oof.items()}

if predicciones:
    nombres={**traduccion_estrategias,'xgb_kmeans5_fijo':'XGBoost + K-means fijo',
             'xgb_kmeans5_grilla30':'XGBoost + K-means 30',
             'xgb_kmeans5_grilla36':'XGBoost + K-means 36'}
    registros=[];cruzados=[];predicciones_cruzadas={}
    pliegues=list(StratifiedKFold(n_splits=5,shuffle=True,random_state=SEMILLA).split(np.zeros(len(y)),y))
    for codigo,p in predicciones.items():
        nombre=nombres.get(codigo,codigo)
        t=elegir_umbral(y,p)
        registros.append(metricas_alerta(y,p,.5,nombre,'Umbral 0,50'))
        pp,rr,tt=precision_recall_curve(y,p)
        f1=2*pp[:-1]*rr[:-1]/np.maximum(pp[:-1]+rr[:-1],1e-12)
        registros.append(metricas_alerta(y,p,float(tt[np.argmax(f1)]),nombre,'Máximo F1 OOF'))
        registros.append(metricas_alerta(y,p,t,nombre,'Recall 98 % OOF'))
        decisiones=np.zeros(len(y),dtype=bool)
        for tr,va in pliegues:
            decisiones[va]=p[va]>=elegir_umbral(y[tr],p[tr])
        predicciones_cruzadas[codigo]=decisiones
        vn,fp,fn,vp=confusion_matrix(y,decisiones,labels=[0,1]).ravel()
        cruzados.append({'Estrategia':nombre,'FN':fn,'FP':fp,
                         'Recall cruzado':vp/(vp+fn),'F1 cruzado':f1_score(y,decisiones),
                         'Exactitud cruzada':accuracy_score(y,decisiones),
                         'Falsas alarmas':fp/(vn+fp),'ROC AUC OOF':roc_auc_score(y,p),
                         'Umbral OOF':t,'Código':codigo})
    tabla_alerta=pd.DataFrame(registros)
    tabla_cruzada=pd.DataFrame(cruzados).sort_values(['Falsas alarmas','FN'])
    display(tabla_alerta[['Estrategia','Política','Umbral','VP','FN','FP','VN','Recall','Precisión','F1','Exactitud','ROC AUC']].round(6))
    display(tabla_cruzada.round(6))
    tabla_alerta.to_csv(SALIDA/'metricas_alerta.csv',index=False)
    tabla_cruzada.to_csv(SALIDA/'metricas_alerta_cruzadas.csv',index=False)
    # Matriz del modelo seleccionado por ROC AUC; las alternativas operativas se mantienen en la tabla.
    elegido=tabla_cruzada.loc[tabla_cruzada['Código']==seleccionada].iloc[0] if seleccionada in predicciones else tabla_cruzada.iloc[0]
    matriz=confusion_matrix(y,predicciones_cruzadas[elegido['Código']],labels=[0,1])
    fig,ax=plt.subplots(figsize=(8,6))
    sns.heatmap(matriz,annot=anotaciones_matriz(matriz),fmt='',cmap='Blues',cbar=False,
       xticklabels=['Legítima predicha','Alerta de fraude'],
       yticklabels=['Legítima real','Fraude real'],ax=ax)
    ax.set(title='Matriz de confusión cruzada · objetivo de exhaustividad 98 %',
           xlabel='Predicción',ylabel='Clase real')
    cerrar_figura(fig,'21_matriz_confusion_umbral')
    fig,ejes=plt.subplots(1,2,figsize=(14,6))
    sns.barplot(data=tabla_cruzada,y='Estrategia',x='FN',color=COLORES[1],ax=ejes[0])
    sns.barplot(data=tabla_cruzada,y='Estrategia',x='Falsas alarmas',color=COLORES[0],ax=ejes[1])
    ejes[0].set(title='Fraudes no detectados',xlabel='Falsos negativos',ylabel='')
    ejes[1].set(title='Tasa de alertas falsas',xlabel='Falsas alarmas / transacciones legítimas',ylabel='')
    cerrar_figura(fig,'22_compromiso_errores_operativos')

### 5.6 Matriz de confusión, F1 y errores operativos

La matriz de confusión informa conteos y porcentajes sobre el conjunto OOF. Estas figuras comparan políticas de decisión y la sensibilidad del F1 ante el umbral; un objetivo de recall del 98 % puede producir un volumen muy elevado de alertas falsas. Los resultados usan predicciones OOF reales, no probabilidades inventadas.


In [ ]:
if predicciones:
    modelo=elegido['Estrategia']
    p_operativo=np.asarray(predicciones[elegido['Código']],dtype=float)
    politicas_modelo=tabla_alerta[tabla_alerta['Estrategia']==modelo]
    fig,ejes=plt.subplots(2,2,figsize=(13.5,9))
    for ax,(col,titulo) in zip(ejes.ravel(),[('F1','F1'),('Precisión','Precisión'),
                                           ('Recall','Exhaustividad'),('Exactitud','Exactitud')]):
        tab=politicas_modelo.sort_values(col,ascending=False)
        sns.barplot(data=tab,x=col,y='Política',color=COLORES[0],ax=ax)
        ax.set(title=titulo+' según la política',xlabel=titulo,ylabel='',xlim=(0,1.02))
    cerrar_figura(fig,'23_metricas_f1_precision_recall_accuracy')
    pp,rr,tt=precision_recall_curve(y,p_operativo,drop_intermediate=True)
    f1=2*pp[:-1]*rr[:-1]/np.maximum(pp[:-1]+rr[:-1],1e-12)
    selec=np.unique(np.linspace(0,len(tt)-1,min(2500,len(tt)),dtype=int))
    fig,ax=plt.subplots(figsize=(12,5.7))
    ax.plot(np.maximum(tt[selec],1e-6),rr[:-1][selec],color=COLORES[1],alpha=.5,label='Exhaustividad')
    ax.plot(np.maximum(tt[selec],1e-6),pp[:-1][selec],color=COLORES[2],alpha=.5,label='Precisión')
    ax.plot(np.maximum(tt[selec],1e-6),f1[selec],color=COLORES[0],alpha=1,lw=2.4,label='F1')
    ax.axvline(tt[np.argmax(f1)],color=COLORES[3],ls='--',label='Máximo F1')
    ax.axvline(float(elegido['Umbral OOF']),color=COLORES[4],ls='--',label='Objetivo 98 %')
    ax.set(xscale='log',title='F1, precisión y exhaustividad según el umbral',
           xlabel='Umbral (escala logarítmica)',ylabel='Métrica (0 a 1)',ylim=(0,1.03))
    ax.legend()
    cerrar_figura(fig,'24_f1_precision_recall_vs_umbral')
    fig,ejes=plt.subplots(1,2,figsize=(15,6))
    f1_ranking=tabla_cruzada.sort_values('F1 cruzado',ascending=False)
    sns.barplot(data=f1_ranking,x='F1 cruzado',y='Estrategia',color=COLORES[0],ax=ejes[0])
    ejes[0].set(title='F1 con umbral seleccionado en otros folds',xlabel='F1 cruzado',ylabel='')
    alertas_ranking=tabla_cruzada.sort_values('FP',ascending=True)
    sns.barplot(data=alertas_ranking,x='FP',y='Estrategia',color=COLORES[1],ax=ejes[1])
    ejes[1].set(title='Alertas sobre operaciones legítimas',xlabel='Falsos positivos',ylabel='')
    cerrar_figura(fig,'25_f1_y_alertas_por_modelo')


## 6. Conclusiones y limitaciones

La comparación distingue efectos de escala, características de agrupamiento y modelo. Los resultados se generan a continuación a partir de las predicciones obtenidas; no se incorporan cifras históricas. Una AUC mayor no implica superioridad estadística. Las métricas calculadas después de ordenar candidatos por AUC OOF son exploratorias. La selección del umbral no constituye una validación prospectiva. La publicación del código debe cumplir las reglas de Kaggle.


In [ ]:
catalogo=pd.DataFrame([{'Figura':p.stem,'Archivo':str(p.relative_to(RAIZ))} for p in sorted(FIGURAS.glob('*.png'))])
display(catalogo)
catalogo.to_csv(SALIDA/'catalogo_figuras.csv',index=False)
resumen_final={'protocolo':'experiment023','seleccion_nominal':seleccionada,
               'n_candidatos':len(resultados_comunes),'n_top10':len(top10),
               'auc_seleccionado':float(top10.iloc[0].roc_auc),
               'observaciones_entrenamiento':len(y),'observaciones_prueba':len(prueba),
               'n_particiones':N_PARTICIONES,'firma':firma,
               'nota':'Selección OOF exploratoria; no equivale a prueba independiente.'}
(SALIDA/'resumen_final.json').write_text(json.dumps(resumen_final,indent=2,ensure_ascii=False),encoding='utf8')
display(pd.DataFrame([resumen_final]))
